[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FabioArnez/LessPrecise-MoreReliable-Tutorial-ICVES-2026-Public/blob/main/notebook/Quantization_Corruption_Robustness_Tutorial.ipynb)

# Quantization noise, strong features, weak features
## Why a network trained under rounding noise can be *more* reliable under corruption

*The hands-on half of the IEEE ICVES 2026 tutorial "Less Precise, More Reliable: Quantization for
Efficient and Robust Road-Scene Perception". Notebook by Aymen Bouguerra, adapted by Fabio Arnez.
The slides, the lecture notes and this notebook are on the [tutorial page](https://fabioarnez.github.io/LessPrecise-MoreReliable-Tutorial-ICVES-2026-Public/).
The references are at the end.*

**Hands-on tutorial, ≈ 1.5 h. Parts 1–4 run on a laptop CPU in seconds. Part 5 needs a GPU (Colab T4, ~12 min, or an Apple-silicon Mac).**

One idea, built up with code and animations:

> Rounding to $b$ bits adds noise of a **fixed size**. A **strong** feature (large
> variance across the data) is barely touched by it; a **weak** feature (small
> variance) drowns in it. A model **trained under that noise** stops relying on the
> weak features. Blur and pixel noise also destroy the weak features first. So the
> model that already stopped relying on them **loses less** when the image degrades.

The toy parts show each step of this idea. Part 5 tests it on a real encoder and marks where the
evidence stops.

| Part | What you see | Runs on |
|---|---|---|
| 1 | Quantization of a 1-D model, one animation per step: FP32 fit · naive 3-bit · calibrated (PTQ) · trained through the quantizer (QAT) | CPU |
| 2 | The same rounding noise on a strong and a weak signal (animated); the survival threshold | CPU |
| 3 | A whole spectrum of features vs the flat noise floor (animated); a classifier trained under the noise moves its decision onto the strong features and becomes robust | CPU |
| 4 | Hard comparison on images: a big, general cue vs a tiny, specialized cue under quantization, blur and noise | CPU |
| 5 | The same measurements on a real vision transformer (DINOv2) under blur | GPU |

Each animation is one cell. Run a cell, watch, read the short note below it, move on.

## Part 0 · Setup

In [ ]:
# Libraries (all preinstalled in Colab)
import os, math, time, copy, collections, warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import animation, gridspec
from IPython.display import HTML, display
from scipy.ndimage import gaussian_filter

# Figure style
plt.rcParams.update({
    "figure.dpi": 105,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "animation.embed_limit": 80,     # MB, so that the longest animations still fit in the notebook
})

# One colour per entity, the same in every figure
C_FP32, C_QAT, C_PTQ, C_NOISE = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"   # methods and noise
C_STRONG, C_MID, C_WEAK = "#2ecc71", "#d4ac0d", "#e74c3c"                     # feature strength

# Fixed seeds, so that every run gives the same numbers
torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

In [ ]:
# @title Animation helpers (setup code, no need to read)
# show_anim() plays a drawing function as an animation with a play button. When the
# environment variable FIG_DIR is set (a check run without a browser), every figure is saved
# as a PNG file instead, and each animation is reduced to a few frames.
SCRIPT_MODE = bool(os.environ.get("FIG_DIR"))
if SCRIPT_MODE:
    os.makedirs(os.environ["FIG_DIR"], exist_ok=True)
    _show, _n = plt.show, [0]

    def _save_show(*args, **kwargs):
        """plt.show() that first saves the current figure as figNN.png."""
        _n[0] += 1
        plt.gcf().savefig(f"{os.environ['FIG_DIR']}/fig{_n[0]:02d}.png", bbox_inches="tight")
        _show(*args, **kwargs)

    plt.show = _save_show


def show_anim(fig, draw, n_frames, interval=60):
    """Animate `fig` by calling draw(0), draw(1), ..., draw(n_frames - 1), one call per frame."""
    if SCRIPT_MODE:                       # save the first, two middle and the last frames
        for k in (0, n_frames // 3, 2 * n_frames // 3):
            draw(k)
            fig.savefig(f"{os.environ['FIG_DIR']}/anim{_n[0] + 1:02d}_f{k:03d}.png", bbox_inches="tight")
        draw(n_frames - 1)
        plt.show()
        return
    anim = animation.FuncAnimation(fig, draw, frames=n_frames, interval=interval, blit=False)
    plt.close(fig)                        # show the animation only, not a static copy of it
    display(HTML(anim.to_jshtml()))

### The six talk clips
Parts 1–3 play six short mp4 clips from the talk. They are **not** inside the
notebook (that made it too heavy for Colab). Run the cell below once:
* **Colab**: the cell downloads `media.zip` (16 MB) from a Google Drive link that
  anyone can open. The Colab machine fetches it directly, so nothing is mounted
  and no permission is requested.
* **local Jupyter**: the clips are found in `media/` next to the notebook.
* **if the download fails**, Colab asks you to upload `media.zip`. You can get the
  file in your browser from
  [this Google Drive link](https://drive.google.com/file/d/15GIPeFZ9FuWeZybVGVmmdzDBJhlJ21kt/view),
  or click *Cancel upload* to continue without the clips (each clip cell then
  prints a note). Setting `MOUNT_DRIVE = True` searches your own Google Drive
  instead, but Google then asks for broad access to your account.

In [ ]:
# @title Load the six talk clips (setup code, no need to read)
import glob
import shutil
import urllib.request
import zipfile
from IPython.display import Video

# Where the clips come from, in the order this cell tries them (see the note above)
MEDIA_ZIP_URL = "https://drive.google.com/uc?export=download&id=15GIPeFZ9FuWeZybVGVmmdzDBJhlJ21kt"
MEDIA_URL = ""          # optional: a web folder that serves the six .mp4 files
MOUNT_DRIVE = False     # Colab: True also searches your own Google Drive (asks for broad access)

CLIPS = [
    "01_fp32_training.mp4",
    "02_naive_3bit.mp4",
    "03_ptq_calibrated.mp4",
    "04_qat.mp4",
    "05_energy_by_rank_noise_floor.mp4",
    "06_strong_vs_weak_feature.mp4",
]
IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False


def _has_clips(folder):
    return all(os.path.exists(os.path.join(folder, clip)) for clip in CLIPS)


def _find_media():
    """A folder that holds all six clips, or None."""
    drive_hits = glob.glob("/content/drive/MyDrive/**/" + CLIPS[0], recursive=True)
    candidates = ["media", ".", "/content/media", "/content"] + [os.path.dirname(f) for f in drive_hits]
    for folder in candidates:
        if _has_clips(folder):
            return folder
    return None


def _unzip(zip_path):
    """Copy the .mp4 files of a zip into media/, whether they sit at top level or in a folder."""
    with zipfile.ZipFile(zip_path) as zf:
        for member in zf.namelist():
            if member.endswith(".mp4"):
                with open(os.path.join("media", os.path.basename(member)), "wb") as f:
                    f.write(zf.read(member))


def _download_zip(url):
    """Download media.zip onto this machine and unzip it. Nothing is mounted, no permission."""
    tmp = "media_download.zip"
    try:
        print("downloading media.zip (16 MB) …")
        with urllib.request.urlopen(url, timeout=120) as response, open(tmp, "wb") as f:
            shutil.copyfileobj(response, f)
        if not zipfile.is_zipfile(tmp):
            raise ValueError("the link did not return a zip file")
        _unzip(tmp)
        return _find_media()
    except Exception as e:
        print("download failed:", e)
        return None
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)


def fetch_media():
    """The folder that holds the clips, or None if every source fails."""
    folder = _find_media()                   # 1. already here (local Jupyter: media/)
    if folder:
        return folder
    os.makedirs("media", exist_ok=True)
    if MEDIA_ZIP_URL:                        # 2. the shared Google Drive link
        folder = _download_zip(MEDIA_ZIP_URL)
        if folder:
            return folder
    if MEDIA_URL:                            # 3. an optional web folder
        try:
            for clip in CLIPS:
                urllib.request.urlretrieve(f"{MEDIA_URL.rstrip('/')}/{clip}", f"media/{clip}")
            return "media"
        except Exception as e:
            print("MEDIA_URL failed:", e)
    if IN_COLAB:
        if MOUNT_DRIVE:                      # 4. your own Google Drive, only if you ask for it
            try:
                from google.colab import drive
                drive.mount("/content/drive", force_remount=False)
                zips = glob.glob("/content/drive/MyDrive/**/media.zip", recursive=True)
                if zips:
                    print("found", zips[0])
                    _unzip(zips[0])
                    return _find_media()
                folder = _find_media()
                if folder:
                    return folder
                print("media.zip not found in Drive")
            except Exception as e:
                print("Drive not available:", e)
        from google.colab import files       # 5. last resort, an upload box
        print("Upload media.zip (the six clips), or click 'Cancel upload' to continue without them …")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                _unzip(name)
            elif name.endswith(".mp4"):
                os.replace(name, os.path.join("media", name))
        return _find_media()
    return None


MEDIA_DIR = None if SCRIPT_MODE else fetch_media()
print("clips:", MEDIA_DIR or "not found (the clip cells will print a note instead)")


def show_clip(name, width=900):
    """Play one of the six clips, or print a note if they are missing."""
    if MEDIA_DIR is None:
        print(f"[clip {name} not available: see 'The six talk clips' in Part 0]")
        return
    display(Video(os.path.join(MEDIA_DIR, name), embed=True, width=width,
                  html_attributes="controls loop"))

## Part 1 · What quantization does to a model, one step at a time

A $b$-bit quantizer with step $s$ maps any real number to one of $2^b$ levels:

$$Q(x) = s \cdot \mathrm{clip}\!\left(\mathrm{round}(x/s),\; -2^{b-1},\; 2^{b-1}-1\right)$$

**Rounding** adds an error of at most $s/2$ (about $s/\sqrt{12}$ RMS).
**Clipping** cuts whatever lies outside the range. The step $s$ is the only knob.

We use an extreme setting so that everything is visible: **3 bits, 8 levels**, applied
to the output of a small network that fits a noisy wave.

In [ ]:
# The toy problem: noisy samples of a smooth wave, which a small network will learn to fit
def truth(x):
    return 4.0 * np.sin(0.5 * x) + 1.5 * np.sin(1.7 * x + 0.8) + 0.8 * np.sin(3.1 * x)


rng = np.random.RandomState(0)
X_DATA = np.sort(rng.uniform(0, 25, 900))                # 900 inputs between 0 and 25
Y_DATA = truth(X_DATA) + 0.6 * rng.randn(len(X_DATA))    # noisy targets
X_GRID = np.linspace(0, 25, 600)                         # where the curves are drawn
Y_TRUE = truth(X_GRID)

# The same data as torch tensors, with the inputs scaled to [0, 1]
xt_data = torch.tensor(X_DATA / 25, dtype=torch.float32)[:, None]
yt_data = torch.tensor(Y_DATA, dtype=torch.float32)[:, None]
xt_grid = torch.tensor(X_GRID / 25, dtype=torch.float32)[:, None]

# The b-bit quantizer of the formula above: round to a multiple of the step, then clip
BITS_TOY = 3
QMIN, QMAX = -(2 ** (BITS_TOY - 1)), 2 ** (BITS_TOY - 1) - 1    # 3 bits give the integers -4 ... 3


def quantize(y, step):
    return step * np.clip(np.round(y / step), QMIN, QMAX)

In [ ]:
# Left: the quantizer applied to every input. Right: the error it makes.
t = np.linspace(-9, 9, 800)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))

ax[0].plot(t, t, ":", color="gray", label="identity (no quantization)")
for step, col in ((2.0, C_PTQ), (1.0, C_QAT)):
    ax[0].plot(t, quantize(t, step), color=col, lw=2, label=f"3-bit, step s = {step}")
ax[0].legend(fontsize=8)
ax[0].set(xlabel="x", ylabel="Q(x)", title="8 levels: round, then clip")

ax[1].plot(t, quantize(t, 2.0) - t, color=C_PTQ, lw=1.5, label="s = 2: coarse rounding, range ±8")
ax[1].plot(t, quantize(t, 1.0) - t, color=C_QAT, lw=1.5, label="s = 1: fine rounding, clipped beyond ±4")
ax[1].axhline(0, color="gray", lw=0.8)
ax[1].legend(fontsize=8)
ax[1].set(xlabel="x", ylabel="Q(x) − x", title="The error: a sawtooth inside the range, linear outside")

plt.tight_layout()
plt.show()

### 1.1 · FP32 training
*Clip 1 from the talk.* Top: a 1→64→64→1 network learns the wave. Middle: the model
error |f(x) − truth| along the input. Bottom: the rounding error, zero for now.

In [ ]:
show_clip("01_fp32_training.mp4")

### 1.2 · Naive 3-bit quantization
*Clip 2.* The trained output is rounded onto 8 levels with a step chosen blindly for a
wide range. The network is unchanged. Watch the two error panels: the model error
triples and a rounding error of up to half a step appears everywhere.

In [ ]:
show_clip("02_naive_3bit.mp4")

### 1.3 · PTQ: calibrate the step
*Clip 3.* Post-training quantization measures the **actual** output range and
shrinks the step so the 8 levels cover just that. Same network, finer staircase,
both errors fall. Still no training.

In [ ]:
show_clip("03_ptq_calibrated.mp4")

### 1.4 · QAT: train through the quantizer
*Clip 4.* The quantizer goes inside the model and training continues. Two tricks:
* **STE** (straight-through estimator): forward = round, backward = identity, so gradients pass through the staircase.
* **LSQ** (learned step size): the step $s$ is a parameter with its own gradient.

The network bends its output so that the *rounded* output fits the data (grey zoom
on the right). The model loss falls toward the floor set by the data noise; the
rounding error stays.

In [ ]:
show_clip("04_qat.mp4")

### 1.5 · The same four stages in code
The four cells below run the same four stages. They train the FP32 network, apply the naive
and the calibrated quantizer, run QAT with STE and LSQ, and draw the four results side by side.

In [ ]:
# Two tricks that let gradients flow through a quantizer

def round_ste(x):
    """Round in the forward pass; pass the gradient through unchanged (straight-through estimator)."""
    return (x.round() - x).detach() + x


def grad_scale(x, g):
    """Return x unchanged in the forward pass; multiply its gradient by g in the backward pass (LSQ)."""
    return (x - x * g).detach() + x * g


class OutputQuantizer(nn.Module):
    """Symmetric b-bit quantizer whose step size s is learned (LSQ, Esser et al., 2020)."""

    def __init__(self, bits, step):
        super().__init__()
        self.qmin, self.qmax = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1
        self.step = nn.Parameter(torch.tensor(float(step)))

    def forward(self, x):
        # keep the step positive, and scale its gradient as LSQ recommends
        s = grad_scale(self.step.clamp(min=1e-4), 1.0 / math.sqrt(x.numel() * self.qmax))
        # divide by the step, clip to the integer range, round (with STE), multiply back
        return round_ste(torch.clamp(x / s, self.qmin, self.qmax)) * s

In [ ]:
def make_mlp():
    """A 1 -> 64 -> 64 -> 1 network with tanh activations, always with the same initial weights."""
    torch.manual_seed(0)
    return nn.Sequential(nn.Linear(1, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh(), nn.Linear(64, 1))


def predict_grid(model):
    """The model's output on X_GRID, as a numpy array."""
    with torch.no_grad():
        return model(xt_grid).squeeze(1).numpy()


# Stage 1: train the network in full precision (FP32)
mlp = make_mlp()
opt = torch.optim.Adam(mlp.parameters(), lr=3e-3)
for it in range(1500):
    loss = F.mse_loss(mlp(xt_data), yt_data)
    opt.zero_grad()
    loss.backward()
    opt.step()
Y_FP32 = predict_grid(mlp)

# Stage 2: naive quantization, with a step chosen for a wide range of ±8
STEP_NAIVE = 8.0 / QMAX
Y_NAIVE = quantize(Y_FP32, STEP_NAIVE)

# Stage 3: post-training quantization (PTQ), with the step calibrated on the actual output range
STEP_PTQ = float(np.abs(Y_FP32).max()) / QMAX
Y_PTQ = quantize(Y_FP32, STEP_PTQ)

In [ ]:
# Stage 4: quantization-aware training (QAT). The quantizer sits inside the model and starts
# from the PTQ step. Training continues, so the network and the step adapt to the rounding.
qmlp = copy.deepcopy(mlp)
quant = OutputQuantizer(BITS_TOY, STEP_PTQ)
opt = torch.optim.Adam([
    {"params": qmlp.parameters(), "lr": 1e-3},
    {"params": [quant.step], "lr": 3e-3},
])
for it in range(900):
    loss = F.mse_loss(quant(qmlp(xt_data)), yt_data)
    opt.zero_grad()
    loss.backward()
    opt.step()

Y_QAT_F = predict_grid(qmlp)            # the QAT network's output before rounding
STEP_QAT = float(quant.step)            # the learned step
Y_QAT = quantize(Y_QAT_F, STEP_QAT)     # what the quantized model outputs

In [ ]:
# The four stages side by side: data (dots), truth (dashes), network (blue), quantized output (orange)
stages = [
    ("1 · FP32", Y_FP32, None, None),
    (f"2 · naive {BITS_TOY}-bit", Y_FP32, Y_NAIVE, STEP_NAIVE),
    ("3 · PTQ (calibrated)", Y_FP32, Y_PTQ, STEP_PTQ),
    ("4 · QAT (trained)", Y_QAT_F, Y_QAT, STEP_QAT),
]
fig, ax = plt.subplots(1, 4, figsize=(16, 3.3), sharey=True)
for a, (name, y, yq, step) in zip(ax, stages):
    a.scatter(X_DATA, Y_DATA, s=3, color="#b0c4de", alpha=0.5)
    a.plot(X_GRID, Y_TRUE, "--", color="#27ae60", lw=1)
    a.plot(X_GRID, y, color=C_FP32, lw=1.6)
    deployed = y if yq is None else yq              # what the model outputs at this stage
    title = f"{name}\nmodel error {np.abs(deployed - Y_TRUE).mean():.2f}"
    if yq is not None:
        a.step(X_GRID, yq, where="mid", color=C_QAT, lw=1.4)
        for k in range(2 ** BITS_TOY):              # the quantization levels
            a.axhline((QMIN + k) * step, color="gray", lw=0.5, ls=":")
        title += f" · rounding error {np.abs(yq - y).mean():.2f} · s = {step:.2f}"
    a.set_title(title, fontsize=9.5)
    a.set_xlim(0, 25)
plt.tight_layout()
plt.show()

**Part 1 in three lines.**
* Naive → PTQ: same network, better step. Calibration alone removes much of the damage.
* PTQ → QAT: the network *changes*. It learns to put its output where the staircase can represent it.
* The rounding error never vanishes. QAT does not remove quantization noise; it teaches the model to **live with it**.
  Part 2 asks *which signals* can live with it.

*Exercise 1.1: set `BITS_TOY = 2` (4 levels) in the first code cell of Part 1, run that cell again,
then run the four code cells of 1.5. The clips of 1.1 to 1.4 do not change.*

## Part 2 · The same noise on a strong feature and a weak feature

Rounding noise has a **fixed size** set by the step. It does not care what it is
added to. So whether a signal survives depends only on the **signal's size relative
to the step**. Extreme example: a sine of amplitude 3 and a sine of amplitude 0.15,
the same small background jitter, the **same** step of 1. Watch the rounding switch on.

*Clip 6 from the talk:* the same rounding on a strong and a weak feature.

In [ ]:
show_clip("06_strong_vs_weak_feature.mp4")

Now the same thing in code, with extreme amplitudes (3 vs 0.15) and the same step
of 1 at the end. The animation grows the step from 0 to 1: the weak feature dies
as soon as the step passes about twice its amplitude; the strong one never does.

In [ ]:
# Two features with very different amplitudes, plus the same small jitter (background noise)
tt = np.linspace(0, 4 * np.pi, 500)
rng = np.random.RandomState(1)
STEP = 1.0          # the rounding step at the end of the animation
JIT = 0.12          # jitter amplitude, the same for both features


def q_step(v, step=STEP):
    """Round v to the nearest multiple of the step (no clipping)."""
    return step * np.round(v / step)


A_STRONG, A_WEAK = 3.0, 0.15
strong = A_STRONG * np.sin(tt) + JIT * rng.randn(len(tt))
weak = A_WEAK * np.sin(tt) + JIT * rng.randn(len(tt))

In [ ]:
# Animation: the step grows from 0 (no rounding) to 1, and both features are rounded with it
STEPS_ANIM = np.concatenate([np.zeros(6), np.logspace(-2, 0, 44), np.ones(10)])

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
panels = (
    (ax[0], strong, A_STRONG, f"strong feature (amplitude {A_STRONG})", C_STRONG, 4),
    (ax[1], weak, A_WEAK, f"weak feature (amplitude {A_WEAK}), y axis zoomed ×4", C_WEAK, 1),
)
lines, lvls = [], []        # per panel, the rounded signal and the quantization levels
for a, sig, amp, name, col, ylim in panels:
    a.plot(tt, amp * np.sin(tt), color="gray", lw=0.9, ls="--", label="true feature")
    ln, = a.step(tt, sig, where="mid", color=col, lw=1.3, label="what the network sees")
    lines.append(ln)
    lvls.append([a.axhline(0, color="gray", lw=0.5, ls=":", alpha=0) for _ in range(9)])
    a.set(ylim=(-ylim, ylim), title=name)
    a.legend(fontsize=8, loc="upper right")
ttl = fig.suptitle("")


def draw_p2(i):
    """Frame i: round both features with the step STEPS_ANIM[i] and update the title."""
    st = STEPS_ANIM[i]
    qs = [q_step(sig, st) if st > 0 else sig for sig in (strong, weak)]
    for ln, q in zip(lines, qs):
        ln.set_ydata(q)
    for levels in lvls:                 # show the levels -4s ... 4s once rounding starts
        for k, h in zip(range(-4, 5), levels):
            if st > 0:
                h.set_ydata([k * st] * 2)
                h.set_alpha(0.7)
            else:
                h.set_alpha(0)
    corr_strong, corr_weak = (np.corrcoef(q, np.sin(tt))[0, 1] for q in qs)
    head = "no rounding" if st == 0 else f"rounding step s = {st:.2f}"
    ttl.set_text(f"{head}    correlation with the true feature "
                 f"(strong {corr_strong:.2f}, weak {corr_weak:.2f})")


show_anim(fig, draw_p2, len(STEPS_ANIM), interval=90)

The strong feature becomes a coarse but faithful staircase. The weak feature, whose
whole amplitude is below **half a step**, is reduced to random flickers driven by the
jitter: the information is gone. **Quantization is a threshold on feature strength.**
Let us measure the threshold.

In [ ]:
# For each bit width, how much of a feature survives rounding, as a function of its amplitude.
# The quantizer covers a fixed range of ±8, as a per-tensor quantizer would, so its step is
# 8 / 2^(b-1). Survival is the correlation between the rounded signal and the clean sine.
amps = np.logspace(-1.5, 1, 60)          # amplitudes from 0.03 to 10
bits_list = [2, 3, 4, 6, 8]

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for b in bits_list:
    step = 8.0 / (2 ** (b - 1))
    corr = []
    for A in amps:
        sig = A * np.sin(tt) + 0.1 * step * rng.randn(len(tt))     # feature plus a small jitter
        qs = np.clip(q_step(sig, step), -8, 8)                     # round, then clip to the range
        corr.append(np.corrcoef(qs, np.sin(tt))[0, 1] if qs.std() > 0 else 0.0)
    ax[0].plot(amps / step, corr, "-", lw=1.8, label=f"{b} bit (step {step:g})")
    ax[1].plot(amps, corr, "-", lw=1.8, label=f"{b} bit")

ax[0].axvline(0.5, color="gray", ls=":")
ax[0].text(0.52, 0.1, "amplitude = half a step", fontsize=8, color="gray")
ax[0].set(xscale="log", xlabel="feature amplitude / quantization step",
          ylabel="correlation kept after rounding",
          title="In units of the step, every bit-width gives the same curve")
ax[0].legend(fontsize=8)
ax[1].set(xscale="log", xlabel="feature amplitude (range fixed to ±8)", ylabel="correlation kept",
          title="In absolute units: more bits move the threshold left")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

**The rule.** A feature survives quantization when its amplitude is a few steps;
it disappears below about half a step. More bits only move the threshold. A real
network pushes *thousands* of features of very different strengths through the
same per-tensor quantizer, so **some are always below the threshold**.

*Interactive (Colab): play with amplitude and bit-width.*

In [ ]:
# Interactive version: move the sliders to change the feature amplitude and the bit width
def show_quant(amplitude=1.0, bits=3):
    """Draw a feature of the given amplitude and its b-bit rounded version (range ±8)."""
    step = 8.0 / (2 ** (bits - 1))
    sig = amplitude * np.sin(tt) + 0.12 * rng.randn(len(tt))
    qs = np.clip(q_step(sig, step), -8, 8)
    fig, ax = plt.subplots(figsize=(9, 2.8))
    ax.plot(tt, sig, color="#999", lw=1, label="signal")
    ax.step(tt, qs, where="mid", color=C_QAT, lw=1.4, label=f"{bits}-bit, step {step:g}")
    ylim = max(4, 1.3 * amplitude)
    ax.set_ylim(-ylim, ylim)
    ax.legend(fontsize=8, loc="upper right")
    ax.set_title(f"correlation kept: {np.corrcoef(qs, np.sin(tt))[0, 1]:.2f}")
    plt.show()


try:
    import ipywidgets as W
    if not SCRIPT_MODE:
        display(W.interactive(
            show_quant,
            amplitude=W.FloatLogSlider(value=1.0, base=10, min=-1.5, max=0.9, step=0.05),
            bits=W.IntSlider(value=3, min=2, max=8, step=1),
        ))
except Exception as e:
    print("ipywidgets unavailable:", e)

## Part 3 · A whole spectrum of features vs one flat noise floor

### 3.1 · Energy by rank and the noise floor
A network's feature vector is not one signal but hundreds. Across a dataset, its
variance is very unevenly spread: a few directions (**ranks**) carry most of the
energy and a long tail carries very little. Rounding noise is **flat**: about
$s^2/12$ of variance in *every* direction. Draw both on one chart and you get a
**noise floor**. Watch it rise as the bit-width goes from 8 down to 2. At low bit widths the
weakest features are smaller than half a step. They are rounded to zero, so their error is the
feature itself, and the floor bends down to follow them.

*Clip 5 from the talk:* energy by feature rank against the noise floor, and how
QAT training pushes energy onto the strongest ranks (the curve under the bars). This shift is in
line with the spectral analysis of the ICML study (Bouguerra et al., 2026a). For
the road-scene encoder of Part 5, the ICVES paper reports an unchanged effective rank (Bouguerra
et al., 2026b).

In [ ]:
show_clip("05_energy_by_rank_noise_floor.mp4")

Now in code. First the floor alone: features with a power-law spectrum, and the
rounding-noise energy per rank as the bit-width goes from 8 down to 2. From here on the step comes
from the largest value seen, as in PTQ calibration, instead of the fixed range of ±8 of Part 2.

In [ ]:
# A synthetic feature vector with D = 48 directions ("ranks"). Its variance falls with the rank
# as a power law, as in real networks, with a few strong directions and a long tail of weak ones.
D, N = 48, 4000
var = 1.0 / np.arange(1, D + 1) ** 1.3          # variance (energy) of each rank
rng = np.random.RandomState(0)
Z = rng.randn(N, D) * np.sqrt(var)               # N samples of the feature vector


def fake_quant(z, bits):
    """Per-tensor symmetric quantizer, with the range set by the largest value seen (calibration).

    Returns the quantized array and the step.
    """
    qmax = 2 ** (bits - 1) - 1
    step = np.abs(z).max() / qmax
    return step * np.clip(np.round(z / step), -qmax - 1, qmax), step


# The noise floor: the energy of the rounding error in each rank, for 2 to 8 bits
FLOOR = {b: ((fake_quant(Z, b)[0] - Z) ** 2).mean(0) for b in range(2, 9)}
ranks = np.arange(1, D + 1)

In [ ]:
# Animation: the bit width falls from 8 to 2 and the noise floor (red dashes) rises.
# A bar is green far above the floor, yellow near it, and red below it.
fig, ax = plt.subplots(figsize=(10, 3.8))
bars = ax.bar(ranks, var, color=C_STRONG, width=0.8, label="feature energy by rank (variance)")
fl, = ax.plot(ranks, FLOOR[8], "--", color="#e74c3c", lw=2,
              label="rounding-noise energy (the noise floor)")
ax.set(yscale="log", ylim=(1e-5, 2), xlabel="feature rank", ylabel="energy")
ax.legend(loc="upper right", fontsize=9)
ttl = ax.set_title("")
bit_seq = [8] * 8 + [7] * 8 + [6] * 8 + [5] * 8 + [4] * 8 + [3] * 8 + [2] * 12   # frames per bit width


def draw_floor(i):
    """Frame i: draw the floor at bit_seq[i] bits and colour each rank by its distance to it."""
    b = bit_seq[i]
    f = FLOOR[b]
    fl.set_ydata(f)
    for bar, v in zip(bars, var):
        if v > 4 * f.mean():
            bar.set_color(C_STRONG)      # far above the floor
        elif v > f.mean():
            bar.set_color(C_MID)         # near the floor
        else:
            bar.set_color(C_WEAK)        # drowned
    n_above = int((var > f.mean()).sum())
    ttl.set_text(f"{b} bit: {n_above} of {D} ranks above the noise floor   "
                 "(green: far above · yellow: near · red: drowned)")


show_anim(fig, draw_floor, len(bit_seq), interval=120)

### 3.2 · What a model does about it when trained under the noise
Synthetic data where we control everything. Features with the spectrum above, two classes:
* a **strong** direction (rank 1) that separates the classes *moderately*;
* a bundle of **weak** directions (ranks 25–48) with a tiny but *clean* class shift: on
  clean data, this is the best evidence there is.

Three linear classifiers on the same inputs: **FP32** trained on exact features,
**PTQ** = the FP32 weights applied to 4-bit quantized features, **QAT** trained on
4-bit quantized features. First, watch **where the decision energy $w_k^2 \cdot \mathrm{var}_k$ goes** during training.

In [ ]:
def make_data(n, c=0.0, seed=0):
    """n samples of the synthetic features, with labels y in {0, 1}.

    The class shifts the strong rank 1 a lot (separable but overlapping) and the 24 weakest
    ranks a little (a tiny but clean signal). c adds flat test-time noise to every rank.
    """
    r = np.random.RandomState(seed)
    y = r.randint(0, 2, n)
    sgn = 2 * y - 1                              # -1 or +1
    Z = r.randn(n, D) * np.sqrt(var)
    Z[:, 0] += 0.70 * sgn                        # the strong direction
    Z[:, 24:] += 0.02 * sgn[:, None]             # the weak bundle (ranks 25 to 48)
    Z += c * r.randn(n, D)                       # corruption of strength c
    return Z.astype(np.float32), y


def train_linear(Z, y, bits=None, steps=800, wd=1e-4, record_every=None):
    """Logistic regression on Z, or on Z quantized to `bits`, trained with Adam.

    Returns the weights, the bias and, if record_every is set, the weights every few steps.
    """
    Zt = torch.tensor(fake_quant(Z, bits)[0] if bits is not None else Z)
    yt = torch.tensor(y, dtype=torch.float32)
    w = torch.zeros(D, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=0.05)
    hist = []
    for s in range(steps):
        loss = F.binary_cross_entropy_with_logits(Zt @ w + b, yt) + wd * (w ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        if record_every and s % record_every == 0:
            hist.append(w.detach().numpy().copy())
    return w.detach().numpy(), float(b.detach()), hist


def accuracy(w, b, Z, y, bits=None):
    """Accuracy in % of the linear classifier (w, b) on Z, quantized to `bits` if given."""
    if bits is not None:
        Z = fake_quant(Z, bits)[0]
    return 100 * float(((Z @ w + b > 0) == y).mean())


def energy(w):
    """Share of the decision carried by each rank, w_k^2 var_k, normalized to sum to 1."""
    return w ** 2 * var / (w ** 2 * var).sum()

In [ ]:
# Train two classifiers on the same 4000 samples: FP32 on exact features, QAT on 4-bit features
BITS_SYN = 4
Ztr, ytr = make_data(4000, seed=0)
W_fp, b_fp, H_fp = train_linear(Ztr, ytr, record_every=5)
W_q, b_q, H_q = train_linear(Ztr, ytr, bits=BITS_SYN, record_every=5)
H_fp, H_q = H_fp[:70], H_q[:70]          # keep the first 350 steps; after that nothing moves

In [ ]:
# Animation: where each classifier puts its decision energy during training
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), sharey=True)
bw = 0.8
b_fp_ = ax[0].bar(ranks, energy(H_fp[1]), bw, color=C_FP32)
b_q_ = ax[1].bar(ranks, energy(H_q[1]), bw, color=C_QAT)
titles = ("FP32: trained on exact features", f"QAT: trained on {BITS_SYN}-bit quantized features")
for a, name in zip(ax, titles):
    a.axvspan(24.5, 48.5, color=C_WEAK, alpha=0.08)              # the weak bundle
    a.text(36, 0.5, "weak bundle\n(below the floor)", ha="center", fontsize=9, color=C_WEAK)
    a.set(xlabel="feature rank", ylim=(0, 0.6), title=name)
ax[0].set_ylabel("share of decision energy  w²·var")
ttl = fig.suptitle("")


def draw_energy(i):
    """Frame i: the decision energy of both classifiers after i * 5 training steps."""
    for bars_, h in ((b_fp_, H_fp[i]), (b_q_, H_q[i])):
        for bar, v in zip(bars_, energy(h)):
            bar.set_height(v)
    weak_fp = 100 * energy(H_fp[i])[24:].sum()
    weak_q = 100 * energy(H_q[i])[24:].sum()
    ttl.set_text(f"training step {i * 5}   share on the weak bundle:  "
                 f"FP32 {weak_fp:.0f}%   QAT {weak_q:.0f}%")


show_anim(fig, draw_energy, len(H_fp), interval=50)

The FP32 classifier pours its decision energy into the weak bundle, because on clean
data that is where the cleanest evidence is. The QAT classifier saw those
directions buried under rounding noise and learned to ignore them: its energy sits on
the strong rank. Now corrupt the test features with flat noise of strength $c$
(Part 5 shows that blur acts like this on real features) and compare the three.

In [ ]:
# Test the three models on fresh data with flat noise of strength c added to every rank.
#   FP32 = the FP32 weights on exact features
#   PTQ  = the same FP32 weights on 4-bit features
#   QAT  = the weights trained on 4-bit features, tested on 4-bit features
cs = np.array([0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.7, 0.9, 1.2, 1.5])
acc = {"FP32": [], "PTQ": [], "QAT": []}
for c in cs:
    Zte, yte = make_data(6000, c=c, seed=1)
    acc["FP32"].append(accuracy(W_fp, b_fp, Zte, yte))
    acc["PTQ"].append(accuracy(W_fp, b_fp, Zte, yte, BITS_SYN))
    acc["QAT"].append(accuracy(W_q, b_q, Zte, yte, BITS_SYN))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for k, col in (("FP32", C_FP32), ("PTQ", C_PTQ), ("QAT", C_QAT)):
    ax[0].plot(cs, acc[k], "-o", color=col, lw=2, ms=4, label=k)
ax[0].set(xlabel="corruption strength c (flat test-time noise)", ylabel="accuracy (%)",
          title="Clean: FP32 wins. Corrupted: QAT wins.")
ax[0].legend(fontsize=9)

d = np.array(acc["QAT"]) - np.array(acc["FP32"])        # gain of QAT over FP32, in points
ax[1].bar(cs, d, width=0.04, color=[C_QAT if v >= 0 else "#c0392b" for v in d])
ax[1].axhline(0, color="gray")
ax[1].set(xlabel="corruption strength c", ylabel="QAT − FP32 (pp)",
          title="A window: a cost on clean data, a gain under corruption")
plt.tight_layout()
plt.show()

**Read it.** On clean data FP32 is better: the weak evidence *is* real evidence. As
the corruption grows, the weak bundle is the first thing to go. FP32, whose large
weights sit exactly there, now amplifies noise and falls fast. QAT never depended on
the bundle and degrades gently. PTQ has FP32's weights *and* the rounding noise: worst
of both. Nothing here mentions images or a particular network. It is a property of
**training under flat noise on features with an uneven spectrum**.

*Exercise 3.1: set `BITS_SYN = 8`. The floor is now far below the bundle. What happens to the window?*
*Exercise 3.2: replace the quantizer by Gaussian noise of variance $s^2/12$. Does the mechanism need rounding, or just noise of the right size?*

*About Exercise 3.2.* In this linear toy, Gaussian noise of the same variance gives a similar
window, because here only the size of the noise matters. A real encoder behaves differently. In
the ICVES paper, a model trained with noise matched to the quantizers' steps, at the same places,
loses about one point under blur where QAT gains two to four (Bouguerra et al., 2026b, Table VII).
So the toy shows how a noise floor pushes a model off weak features, but not that the size of the
floor explains the gain. Part 5 comes back to this.

## Part 4 · Hard comparison on images: a general cue vs a specialized cue

Now the same story with things you can *see*. Each 32×32 image contains
* a **general** cue: a big bright blob. Its position (left / right) tells the class,
  but with overlap, so it is only ~90% reliable. Large, bright, low-frequency:
  **high variance across the dataset**.
* a **specialized** cue: a small striped patch in the corner. The stripe
  *orientation* (vertical / horizontal) tells the class almost perfectly. Fine
  detail, low contrast, high-frequency: **low variance**, but the best evidence there is.

We put both cues through **3-bit quantization**, **blur** and **pixel noise** and
look at what survives.

In [ ]:
# Synthetic 32 x 32 images with two cues for the class y in {0, 1}
S = 32
YY, XX = np.mgrid[0:S, 0:S]                       # pixel coordinates
PATCH = slice(2, 8)                               # the 6 x 6 corner patch with the stripes
py, px = np.mgrid[0:6, 0:6]
AMP = 0.07                                        # stripe contrast
STRIPES_V = 0.24 + AMP * np.where(px % 2 == 0, 1, -1)     # vertical stripes, class 0
STRIPES_H = 0.24 + AMP * np.where(py % 2 == 0, 1, -1)     # horizontal stripes, class 1
BITS_IMG = 3                                      # bit width of the quantized images


def make_images(n, seed, blur=0.0, noise=0.0):
    """n images and their labels, with optional Gaussian blur (sigma in pixels) and extra noise."""
    r = np.random.RandomState(seed)
    y = r.randint(0, 2, n)
    X = np.zeros((n, S, S), np.float32)
    for i in range(n):
        # general cue: a big bright blob, left for class 0 and right for class 1, with overlap
        cx = np.clip(r.normal(11 if y[i] == 0 else 21, 4), 7, 25)
        cy = r.normal(18, 2)
        X[i] = 0.8 * np.exp(-((XX - cx) ** 2 + (YY - cy) ** 2) / (2 * 6.0 ** 2))
        # specialized cue: the stripe orientation in the corner patch
        X[i, PATCH, PATCH] += STRIPES_V if y[i] == 0 else STRIPES_H
    if blur > 0:                                   # optics: blur happens before the sensor
        X = np.stack([gaussian_filter(a, blur) for a in X])
    X += (0.01 + noise) * r.randn(*X.shape)        # sensor noise, always present, optionally more
    return X.astype(np.float32), y


def fq_img(X, bits):
    """Quantize images to `bits`, with the range set by the largest pixel value."""
    q = 2 ** (bits - 1) - 1
    s = np.abs(X).max() / q
    return (s * np.clip(np.round(X / s), -q - 1, q)).astype(np.float32)


def stripe_contrast(im, y):
    """Mean stripe modulation inside the patch, along the stripe orientation of class y."""
    sign = np.where((px if y == 0 else py) % 2 == 0, 1, -1)
    return float((im[PATCH, PATCH] * sign).mean())

In [ ]:
# One image as each model sees it: exact, quantized to 3 bits, blurred, and with pixel noise
X0, y0 = make_images(1, 4)
img = X0[0]
views = [
    ("FP32 image", img),
    (f"{BITS_IMG}-bit quantized", fq_img(X0, BITS_IMG)[0]),
    ("blur σ = 1.5", gaussian_filter(img, 1.5)),
    ("pixel noise 0.1", img + 0.1 * np.random.RandomState(0).randn(S, S)),
]
fig, ax = plt.subplots(2, 4, figsize=(13, 7))
for j, (name, im) in enumerate(views):
    # top row: the whole image, with the stripe patch outlined in red
    ax[0, j].imshow(im, cmap="gray", vmin=0, vmax=0.9)
    ax[0, j].set_title(name)
    ax[0, j].axis("off")
    ax[0, j].grid(False)
    ax[0, j].add_patch(plt.Rectangle((1.5, 1.5), 6, 6, fill=False, color=C_WEAK, lw=1.5))
    # bottom row: a zoom on the patch
    ax[1, j].imshow(im[0:10, 0:10], cmap="gray", vmin=0, vmax=0.5, interpolation="nearest")
    ax[1, j].axis("off")
    ax[1, j].grid(False)
    ax[1, j].set_title(f"zoom on the patch\nstripe contrast {stripe_contrast(im, y0[0]):.3f}", fontsize=9)
plt.suptitle("The blob (general cue) is untouched by all three. "
             "The stripes (specialized cue) are erased by all three.", y=0.995)
plt.tight_layout(h_pad=2.5)
plt.show()

### 4.1 · Measure the survival of each cue
For each cue, a matched filter: correlate the image with the cue's template and ask
how well that score alone separates the two classes ($d'$ = gap between class means
over the spread). Clean, and after each corruption.

In [ ]:
# A matched filter for each cue: correlate the image with the cue's template
tpl_blob = 0.8 * np.exp(-((XX - 16) ** 2 + (YY - 16) ** 2) / (2 * 6.0 ** 2)) * np.sign(XX - 16)
tpl_blob /= np.abs(tpl_blob).sum()                  # left-versus-right blob detector
tpl_stripes = np.zeros((S, S))
tpl_stripes[PATCH, PATCH] = STRIPES_H - STRIPES_V   # stripe orientation detector


def dprime(X, y, tpl):
    """Class separation d' of the filter score: gap between the class means over the spread."""
    sc = X.reshape(len(X), -1) @ tpl.ravel()
    gap = abs(sc[y == 1].mean() - sc[y == 0].mean())
    return gap / np.sqrt(0.5 * (sc[y == 1].var() + sc[y == 0].var()))


# d' of both cues on clean images and after each corruption
corr_list = [
    ("clean", dict()),
    ("2-bit", dict(bits=2)),
    ("3-bit", dict(bits=3)),
    ("4-bit", dict(bits=4)),
    ("blur σ=1", dict(blur=1)),
    ("blur σ=2", dict(blur=2)),
    ("noise 0.1", dict(noise=0.1)),
    ("noise 0.2", dict(noise=0.2)),
]
db, dm = [], []                                     # d' of the blob and of the stripes
for name, kw in corr_list:
    Xc, yc = make_images(1500, 5, **{k: v for k, v in kw.items() if k != "bits"})
    if "bits" in kw:
        Xc = fq_img(Xc, kw["bits"])
    db.append(dprime(Xc, yc, tpl_blob))
    dm.append(dprime(Xc, yc, tpl_stripes))

fig, ax = plt.subplots(figsize=(11, 3.8))
xs_ = np.arange(len(corr_list))
ax.bar(xs_ - 0.2, db, 0.4, color=C_STRONG, label="general cue (big blob)")
ax.bar(xs_ + 0.2, dm, 0.4, color=C_WEAK, label="specialized cue (stripe orientation)")
ax.set_xticks(xs_)
ax.set_xticklabels([n for n, _ in corr_list])
ax.set(yscale="log", ylabel="class separation d′ of the cue alone (log)",
       title="Quantization, blur and noise all remove the same thing: the weak, specialized cue")
ax.legend(fontsize=9)
plt.show()
d_table = {n: (round(float(a), 2), round(float(b), 2)) for (n, _), a, b in zip(corr_list, db, dm)}
print("d′ of each cue (blob, stripes):", d_table)

Three very different operations, one outcome: the high-variance cue keeps its
$d'$ ≈ 2.8 under everything; the low-variance cue goes from $d'$ ≈ 37 to below 1
at 3 and 2 bits, and to between 1.6 and 5.5 under blur or noise. (At 4 bit the stripes
still fit inside the levels, so they survive: the threshold of Part 2 again.)
That is why a model that learned to stop relying on low-variance cues *for one
reason* (rounding noise) is also protected against the *other* ones (blur, noise).

### 4.2 · What does each classifier bet on?
A linear classifier on pixels has a weight image we can look at. Train one on exact
images (FP32) and one on 3-bit quantized images (QAT), and measure the share of
decision energy $w^2 \cdot \mathrm{var}$ that sits on the patch (the same quantity
as in Part 3).

In [ ]:
def train_pix(X, y, bits=None, steps=600, wd=1e-4):
    """Logistic regression on the raw pixels, or on pixels quantized to `bits`."""
    Xt = torch.tensor((fq_img(X, bits) if bits else X).reshape(len(X), -1))
    yt = torch.tensor(y, dtype=torch.float32)
    w = torch.zeros(S * S, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    opt = torch.optim.Adam([w, b], lr=0.02)
    for _ in range(steps):
        loss = F.binary_cross_entropy_with_logits(Xt @ w + b, yt) + wd * (w ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return w.detach().numpy(), float(b.detach())


def acc_pix(w, b, X, y, bits=None):
    """Accuracy in % of the pixel classifier, on images quantized to `bits` if given."""
    X = fq_img(X, bits) if bits else X
    return 100 * float(((X.reshape(len(X), -1) @ w + b > 0) == y).mean())


def patch_share(w, X):
    """Share of the decision energy w² · var that sits on the patch pixels.

    var is measured on the images the model sees (exact or quantized).
    """
    e = (w ** 2 * X.reshape(len(X), -1).var(0)).reshape(S, S)
    return e[PATCH, PATCH].sum() / e.sum()

In [ ]:
# Train on 3000 images (FP32 on exact pixels, QAT on 3-bit pixels) and look at the weights
Xtr_i, ytr_i = make_images(3000, 0)
Wp_fp, bp_fp = train_pix(Xtr_i, ytr_i)
Wp_q, bp_q = train_pix(Xtr_i, ytr_i, BITS_IMG)

fig, ax = plt.subplots(1, 2, figsize=(9, 4))
panels = (
    (ax[0], Wp_fp, Xtr_i, "FP32 weight image", C_FP32),
    (ax[1], Wp_q, fq_img(Xtr_i, BITS_IMG), f"QAT ({BITS_IMG}-bit) weight image", C_QAT),
)
for a, w, X_seen, name, col in panels:
    m = np.abs(w).max()
    a.imshow(w.reshape(S, S), cmap="RdBu_r", vmin=-m, vmax=m)    # red positive, blue negative
    a.axis("off")
    a.grid(False)
    a.add_patch(plt.Rectangle((1.5, 1.5), 6, 6, fill=False, color="k", lw=1, ls=":"))
    share = 100 * patch_share(w, X_seen)
    a.set_title(f"{name}\n{share:.0f}% of the decision energy on the patch", color=col)
plt.suptitle("FP32 bets on the stripes. QAT, which never saw them, bets on the blob.", y=1.0)
plt.tight_layout()
plt.show()

In [ ]:
# Test the three pixel classifiers under blur, pixel noise, and both
tests = [                                # (name, blur sigma, extra pixel noise)
    ("clean", 0, 0),
    ("blur 1", 1, 0),
    ("blur 2", 2, 0),
    ("blur 3", 3, 0),
    ("noise 0.1", 0, 0.1),
    ("noise 0.2", 0, 0.2),
    ("blur 2 + noise 0.1", 2, 0.1),
    ("blur 2 + noise 0.2", 2, 0.2),
]
res = {"FP32": [], "PTQ": [], "QAT": []}
for name, bl, nz in tests:
    Xte_i, yte_i = make_images(3000, 1, blur=bl, noise=nz)
    res["FP32"].append(acc_pix(Wp_fp, bp_fp, Xte_i, yte_i))
    res["PTQ"].append(acc_pix(Wp_fp, bp_fp, Xte_i, yte_i, BITS_IMG))      # FP32 weights, 3-bit pixels
    res["QAT"].append(acc_pix(Wp_q, bp_q, Xte_i, yte_i, BITS_IMG))

labels = [t[0] for t in tests]
xs_ = np.arange(len(tests))
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
# left: the accuracy of each model under each corruption
for k, (name, col) in enumerate((("FP32", C_FP32), ("PTQ", C_PTQ), ("QAT", C_QAT))):
    ax[0].bar(xs_ + (k - 1) * 0.27, res[name], 0.27, color=col, label=name)
ax[0].set_xticks(xs_)
ax[0].set_xticklabels(labels, rotation=20, fontsize=8)
ax[0].set(ylabel="accuracy (%)", ylim=(50, 101), title="Accuracy under each corruption")
ax[0].legend(fontsize=8)
# right: what each model loses relative to its own clean accuracy
for name, col in (("FP32", C_FP32), ("QAT", C_QAT)):
    ax[1].plot(xs_, np.array(res[name]) - res[name][0], "-o", color=col, lw=2, label=name)
ax[1].set_xticks(xs_)
ax[1].set_xticklabels(labels, rotation=20, fontsize=8)
ax[1].axhline(0, color="gray")
ax[1].set(ylabel="change from clean (pp)",
          title="What each model *loses* when the specialized cue is destroyed")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()
print("FP32 %s\nQAT  %s" % (np.round(res["FP32"], 1), np.round(res["QAT"], 1)))

**Read it.** Under blur alone and under noise alone, FP32 stays ahead of QAT at every level.
QAT loses less relative to its own clean accuracy, and it overtakes FP32 only when blur and noise
combine (83.2% against 81.0%, and 77.5% against 71.6%).

### 4.3 · The dial: coarser quantization → more reliance on the general cue → more robust
Train QAT classifiers at 8, 6, 5, 4, 3 and 2 bits. For each, split the decision
energy $w^2 \cdot \mathrm{var}$ between the **specialized** cue (patch pixels) and
the **general** cue (everything else, i.e. the blob), and test under blur and noise.

In [ ]:
# One QAT pixel classifier per bit width (None = FP32). For each, measure the share of its
# decision energy on the specialized cue, and its accuracy clean, blurred, and blurred with noise.
BITS_SWEEP = [None, 8, 6, 5, 4, 3, 2]
Xte_clean, yte_clean = make_images(3000, 1)
Xte_blur, yte_blur = make_images(3000, 1, blur=2)
Xte_bn, yte_bn = make_images(3000, 1, blur=2, noise=0.2)

share_spec, share_gen, acc_c, acc_b, acc_bn = [], [], [], [], []
for bts in BITS_SWEEP:
    w_, b_ = train_pix(Xtr_i, ytr_i, bts)
    seen = fq_img(Xtr_i, bts) if bts else Xtr_i        # the images this model was trained on
    ps = patch_share(w_, seen)
    share_spec.append(100 * ps)
    share_gen.append(100 * (1 - ps))
    acc_c.append(acc_pix(w_, b_, Xte_clean, yte_clean, bts))
    acc_b.append(acc_pix(w_, b_, Xte_blur, yte_blur, bts))
    acc_bn.append(acc_pix(w_, b_, Xte_bn, yte_bn, bts))

labels = ["FP32"] + [f"{b} bit" for b in BITS_SWEEP[1:]]
xs_ = np.arange(len(labels))
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
# left: what each model relies on
ax[0].bar(xs_, share_gen, color=C_STRONG,
          label="general cue (blob): survives quantization, blur, noise")
ax[0].bar(xs_, share_spec, bottom=share_gen, color=C_WEAK,
          label="specialized cue (stripes): dies under all three")
ax[0].set_xticks(xs_)
ax[0].set_xticklabels(labels)
ax[0].set(ylabel="share of decision energy (%)",
          title="What the model relies on, as the quantization gets coarser")
ax[0].legend(fontsize=8, loc="lower left")
# right: accuracy, clean and corrupted
curves = (
    (acc_c, "#555", "clean"),
    (acc_b, C_FP32, "blur σ = 2"),
    (acc_bn, C_NOISE, "blur σ = 2 + noise 0.2"),
)
for v, col, lab in curves:
    ax[1].plot(xs_, v, "-o", color=col, lw=2, label=lab)
ax[1].set_xticks(xs_)
ax[1].set_xticklabels(labels)
ax[1].set(ylabel="accuracy (%)", title="Clean accuracy drops, accuracy under blur plus noise rises")
ax[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

for l, a_, b_, c_, sp in zip(labels, acc_c, acc_b, acc_bn, share_spec):
    print(f"{l:>6}: specialized share {sp:5.1f}%   "
          f"clean {a_:5.1f}   blur {b_:5.1f}   blur+noise {c_:5.1f}")

**What the dial shows.** Down to 4 bits the rounding step is below the stripe contrast, so the
model still bets on the stripes and stays fragile. At 3 bits the stripes fall under the noise floor
(the threshold of Part 2). The decision energy jumps from 93% on the stripes to 95% on the blob,
accuracy under blur plus noise rises, and clean accuracy falls. Under blur alone, accuracy stays
within 3 points of FP32 at every bit width. At 2 bits the step is so coarse that the blob itself is
damaged and everything drops again, which is the window of Part 3 seen along the bit axis. In this
toy, quantization removed the fragile cue from training, so the model had to learn the robust one.

In a deep network the "stripes" are not pixels but low-variance feature directions. Part 5 checks,
on a real encoder, whether rounding and blur damage the same directions. Keep one limit in mind. The
toy shows how a noise floor can push a model off weak features. It does not show that the floor
alone explains the gain of a real network, which Part 5 also discusses.

## Part 5 · The real thing: a vision transformer under blur

A frozen **DINOv2 ViT-S/14** backbone, a linear head trained on road-object crops
from the **DAWN** dataset (fog, rain, sand, snow). We quantize the 48 linear layers
of the transformer to **8-bit weights and activations** (W8A8): first by calibration
(PTQ), then by training the activation steps and the head with the quantizers in the
loop (QAT), on **clean crops only**. Then we blur the test crops with increasing σ.

Same objects as Part 1 (STE rounding, LSQ steps), only many of them.

> **Runtime.** A GPU is required, a Colab T4 (about 10 to 15 min) or the GPU of an
> Apple-silicon Mac (about 8 min). On a CPU alone Part 5 runs for hours, so read the code
> and follow the presenter's run.

In [ ]:
# Libraries for Part 5. Colab has timm and datasets already; elsewhere this installs them once.
import subprocess
import sys
try:
    import timm, datasets  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm>=0.9", "datasets>=2.18"],
                   check=True)
import timm
from PIL import Image, ImageFilter
from datasets import load_dataset, concatenate_datasets

# Run on a GPU: CUDA on Colab, Apple's GPU (MPS) on a Mac, otherwise the CPU (very slow)
if torch.cuda.is_available():
    DEV = "cuda"
elif torch.backends.mps.is_available():
    DEV = "mps"
else:
    DEV = "cpu"
print("device:", DEV)

# Settings
SEED = 0              # one seed for the whole experiment
QAT_STEPS = 400       # QAT training steps (the FP32 control gets the same extra budget)
HEAD_STEPS = 3000     # training steps of the linear head on frozen FP32 features
BATCH = 32            # batch size for training and inference
QUICK = False         # set True for 700 test crops instead of about 1460 (2x faster, noisier)
CLASSES = ["Car", "Truck", "Bus", "Person"]
IMG_SIZE = 224        # every crop is resized to 224 x 224 pixels
MIN_SIDE = 40         # objects whose shorter side is under 40 pixels are skipped
CAR_TRAIN_CAP = 700   # at most 700 car crops in training, because cars dominate DAWN
DAWN_REVISION = "fd7a5b87f8e31daf255dcc4ddf2642f3ad9744bf"   # the DAWN version tested here

**The data.** DAWN has 1027 road images taken in fog, rain, sand and snow, with a box around
each object. The next two cells cut a square crop around every car, truck, bus and person
whose shorter side is at least 40 pixels, and resize it to 224 × 224 pixels. A quarter of the
images, with all their crops, form the test set, so that no image is in both sets.

In [ ]:
def build_crops():
    """Load DAWN and cut one square crop per object.

    Returns the crops (uint8, N x 224 x 224 x 3), their class indices, the id of the image each
    crop comes from, and the number of images.
    """
    ds = concatenate_datasets([
        load_dataset("Maxim37/dawn-dataset", split="train", revision=DAWN_REVISION),
        load_dataset("Maxim37/dawn-dataset", split="val", revision=DAWN_REVISION),
    ])
    xs, ys, img_ids = [], [], []
    for ex in ds:
        im = ex["image"].convert("RGB")
        Wd, Hd = im.size
        for o in ex["objects"]:
            # keep the four classes, and skip objects that are too small
            if o["class_name"] not in CLASSES or min(o["width"], o["height"]) < MIN_SIDE:
                continue
            # a square around the object's centre, 10% larger than its longer side, inside the image
            cx = o["x_min"] + o["width"] / 2
            cy = o["y_min"] + o["height"] / 2
            side = max(o["width"], o["height"]) * 1.1
            box = (
                int(np.clip(cx - side / 2, 0, Wd - 1)),
                int(np.clip(cy - side / 2, 0, Hd - 1)),
                int(np.clip(cx + side / 2, 1, Wd)),
                int(np.clip(cy + side / 2, 1, Hd)),
            )
            crop = im.crop(box).resize((IMG_SIZE, IMG_SIZE), Image.BICUBIC)
            xs.append(np.asarray(crop, dtype=np.uint8))
            ys.append(CLASSES.index(o["class_name"]))
            img_ids.append(ex["image_id"])
    return np.stack(xs), np.array(ys), np.array(img_ids), len(ds)

In [ ]:
def split_by_image(y, img_ids, frac_test=0.25, seed=0):
    """Indices of the training crops and of the test crops.

    A quarter of the images go to the test set with all their crops. In training, the car crops
    are capped at CAR_TRAIN_CAP so that the four classes are less unbalanced.
    """
    r = np.random.RandomState(seed)
    uniq = np.unique(img_ids)
    r.shuffle(uniq)
    test_imgs = set(uniq[:int(len(uniq) * frac_test)])
    te = np.array([i in test_imgs for i in img_ids])
    tr_idx, te_idx = np.where(~te)[0], np.where(te)[0]
    car = CLASSES.index("Car")
    car_tr = tr_idx[y[tr_idx] == car]
    r.shuffle(car_tr)
    kept = set(car_tr[:CAR_TRAIN_CAP]) | set(tr_idx[y[tr_idx] != car])
    return np.array(sorted(kept)), te_idx


t0 = time.time()
X_u8, Y, IMG_IDS, N_IMAGES = build_crops()
TR, TE = split_by_image(Y, IMG_IDS)
if QUICK:
    TE = np.random.RandomState(0).choice(TE, 700, replace=False)
print(f"{len(Y)} object crops from {N_IMAGES} images in {time.time()-t0:.0f}s | "
      f"train {len(TR)} / test {len(TE)} (split by image, Car capped in train)")

**Blur.** The training crops stay clean. Every test crop is blurred at eight levels, from
σ = 0 (clean) to σ = 14 pixels, so that each model is tested on the same crops at every level.

In [ ]:
# The network input: pixels scaled to [0, 1], then normalized with the ImageNet mean and std
MEAN = torch.tensor((0.485, 0.456, 0.406)).view(1, 3, 1, 1)
STD = torch.tensor((0.229, 0.224, 0.225)).view(1, 3, 1, 1)


def to_input(u8):
    """uint8 crops (N x H x W x 3) to a normalized float tensor (N x 3 x H x W) on the device."""
    x = torch.as_tensor(np.ascontiguousarray(u8)).permute(0, 3, 1, 2).float().div_(255)
    return ((x - MEAN) / STD).to(DEV)


def blur_all(x_u8, sigma):
    """Gaussian blur with standard deviation sigma (in pixels) on every crop; 0 leaves them clean."""
    if sigma == 0:
        return x_u8
    return np.stack([
        np.asarray(Image.fromarray(a).filter(ImageFilter.GaussianBlur(float(sigma))), dtype=np.uint8)
        for a in x_u8
    ])


SIGMAS = [0, 1, 2, 4, 6, 8, 10, 14]
TR_X = X_u8[TR]                                   # clean training crops
TR_Y = torch.as_tensor(Y[TR])
TE_Y = torch.as_tensor(Y[TE])
X_te = X_u8[TE]                                   # clean test crops
TE_CELLS = {s: blur_all(X_te, s) for s in SIGMAS}  # the test crops at every blur level

In [ ]:
# A running example: the bus or truck with the most visible structure, through the blur sweep
def structure(a):
    """Edge energy after a 3-pixel blur: high for a large, readable object."""
    g = np.asarray(Image.fromarray(a).convert("L").filter(ImageFilter.GaussianBlur(3)), dtype=np.float32)
    return float(np.mean(np.diff(g, axis=1)[:-1] ** 2 + np.diff(g, axis=0)[:, :-1] ** 2))


cand = np.where(np.isin(Y[TE], [CLASSES.index("Bus"), CLASSES.index("Truck")]))[0]
EXAMPLE = int(cand[np.argmax([structure(X_te[j]) for j in cand])])

fig, ax = plt.subplots(1, len(SIGMAS), figsize=(15, 2.3))
for a, s in zip(ax, SIGMAS):
    a.imshow(TE_CELLS[s][EXAMPLE])
    a.set_title("clean" if s == 0 else f"σ = {s}", fontsize=9)
    a.axis("off")
    a.grid(False)
plt.suptitle(f"One test {CLASSES[Y[TE][EXAMPLE]].lower()} through the blur sweep", y=1.05)
plt.show()

**The FP32 control.** A frozen DINOv2 ViT-S/14 backbone turns each crop into 384 features,
and a linear head maps them to the four classes. Only the head is trained.

In [ ]:
def make_backbone():
    """A fresh DINOv2 ViT-S/14 with its pretrained weights and no classification head."""
    return timm.create_model("vit_small_patch14_dinov2.lvd142m", pretrained=True,
                             num_classes=0, img_size=IMG_SIZE)


class Classifier(nn.Module):
    """The backbone followed by a linear head."""

    def __init__(self, backbone, head):
        super().__init__()
        self.backbone, self.head = backbone, head

    def forward(self, x):
        return self.head(self.backbone(x))


@torch.inference_mode()
def extract_feats(backbone, xt):
    """Backbone features of uint8 crops, computed in batches and returned on the CPU."""
    backbone.eval()
    return torch.cat([backbone(to_input(xt[b:b + BATCH])).cpu() for b in range(0, len(xt), BATCH)])


def train_head_on_feats(head, feats, y, steps, batch, lr=1e-3, seed=SEED):
    """Train a linear head on precomputed features with AdamW and random mini-batches."""
    g = torch.Generator().manual_seed(seed)          # a fixed order of mini-batches
    head = head.to(DEV).train()
    opt = torch.optim.AdamW(head.parameters(), lr=lr)
    for _ in range(steps):
        idx = torch.randint(0, len(y), (batch,), generator=g)
        loss = F.cross_entropy(head(feats[idx].to(DEV)), y[idx].to(DEV))
        opt.zero_grad()
        loss.backward()
        opt.step()
    return head.eval()

In [ ]:
# Head C learns on frozen FP32 features for 3000 steps. The FP32 control then trains C for
# 400 more steps, the budget QAT gets later, so that the comparison is fair.
t0 = time.time()
FP_BACKBONE = make_backbone().to(DEV).eval()
TR_FEATS = extract_feats(FP_BACKBONE, TR_X)
torch.manual_seed(SEED)                                   # the head's initial weights
HEAD_C = train_head_on_feats(nn.Linear(384, 4), TR_FEATS, TR_Y, HEAD_STEPS, batch=256)
FP_HEAD = train_head_on_feats(copy.deepcopy(HEAD_C), TR_FEATS, TR_Y, QAT_STEPS, batch=BATCH)
FP32 = Classifier(FP_BACKBONE, FP_HEAD).to(DEV).eval()
print(f"FP32 control ready in {time.time()-t0:.0f}s")

### 5.1 · The quantizers, now inside a transformer
Every `nn.Linear` of the 12 blocks gets a **per-tensor** LSQ activation quantizer
(calibrated on 256 clean crops, then learnable) and a **per-channel** weight quantizer
with STE. Balanced accuracy (mean per-class recall) is the metric, because cars dominate.

In [ ]:
# The quantizer building blocks. This cell needs only torch, so you can copy it into your own
# project. round_ste and grad_scale are the two tricks of Part 1.5, repeated here.

def round_ste(x):
    """Round in the forward pass; pass the gradient through unchanged (straight-through estimator)."""
    return (x.round() - x).detach() + x


def grad_scale(x, g):
    """Return x unchanged in the forward pass; multiply its gradient by g in the backward pass (LSQ)."""
    return (x - x * g).detach() + x * g


class ActQuantizer(nn.Module):
    """Per-tensor symmetric quantizer for activations, with a learnable step (LSQ).

    It has three modes:
      calibrating = True  pass x through and record a running average of max |x|;
      ready = False       pass x through (the quantizer is not set up yet);
      otherwise           quantize x with the step `scale` (learned during QAT).
    """

    def __init__(self, bits=8):
        super().__init__()
        self.qmin, self.qmax = -(2 ** (bits - 1)), 2 ** (bits - 1) - 1    # -128 ... 127 at 8 bits
        self.scale = nn.Parameter(torch.ones(1))                         # the step size
        self.register_buffer("amax", torch.tensor(0.0))                  # calibrated max |x|
        self.calibrating = False
        self.ready = False

    def forward(self, x):
        if self.calibrating:
            with torch.no_grad():
                batch_max = x.abs().max()
                if self.amax > 0:
                    self.amax.copy_(0.9 * self.amax + 0.1 * batch_max)   # running average
                else:
                    self.amax.copy_(batch_max)                           # first batch
            return x
        if not self.ready:
            return x
        s = self.scale.clamp(min=1e-8)
        if self.training:
            s = grad_scale(s, 1.0 / math.sqrt(x.numel() * self.qmax))     # LSQ gradient scale
        return round_ste(torch.clamp(x / s, self.qmin, self.qmax)) * s


def quantize_weight(w, bits=8):
    """Per-channel symmetric weight quantization (one step per output row), with STE.

    The step is fixed by max |w| of each row; it is not learned.
    """
    qmax = 2 ** (bits - 1) - 1
    s = (w.abs().amax(dim=1, keepdim=True) / qmax).clamp(min=1e-8)
    w_q = torch.clamp(torch.round(w / s), -qmax - 1, qmax) * s
    return w + (w_q - w).detach()          # forward: quantized weights; backward: identity


class QLinear(nn.Module):
    """A Linear layer that quantizes its input (ActQuantizer) and its weights (quantize_weight)."""

    def __init__(self, lin, w_bits=8, a_bits=8):
        super().__init__()
        self.lin, self.w_bits, self.aq = lin, w_bits, ActQuantizer(a_bits)

    def forward(self, x):
        return F.linear(self.aq(x), quantize_weight(self.lin.weight, self.w_bits), self.lin.bias)

In [ ]:
def quantize_backbone(backbone, w_bits=8, a_bits=8):
    """Replace every nn.Linear inside the transformer blocks by a QLinear, in place.

    In DINOv2 ViT-S that is 4 layers per block (attention qkv and proj, MLP fc1 and fc2) in
    12 blocks, so 48 layers. The patch embedding, the norms and the attention products stay
    in floating point.
    """
    n = 0
    for name, mod in list(backbone.named_modules()):
        if isinstance(mod, nn.Linear) and "blocks" in name:
            # find the parent module, for example "blocks.3.mlp.fc1" -> backbone.blocks[3].mlp
            parent = backbone
            for p in name.split(".")[:-1]:
                parent = parent[int(p)] if p.isdigit() else getattr(parent, p)
            setattr(parent, name.split(".")[-1], QLinear(mod, w_bits, a_bits))
            n += 1
    print(f"  quantized {n} Linear layers → W{w_bits}A{a_bits}")
    return backbone

**Calibration and QAT.** PTQ only calibrates. It runs 256 clean training crops through the
network and sets each activation step from a running average of the largest values seen. QAT
starts from the same calibration, then trains the 48 steps and the head for 400 steps with the
quantizers in the loop. The backbone weights stay frozen in both. This QAT differs from the toys
of Parts 3 and 4, where the whole classifier was trained on quantized inputs. The bit widths are
`W_BITS` and `A_BITS` in the cell that builds the two models.

In [ ]:
@torch.inference_mode()
def calibrate(model, xt, n_batches=8):
    """PTQ calibration on the first 8 batches of crops: each step becomes max |x| / qmax."""
    aqs = [m for m in model.modules() if isinstance(m, ActQuantizer)]
    for m in aqs:
        m.calibrating = True
    model.eval()
    for b in range(n_batches):
        model(to_input(xt[b * BATCH:(b + 1) * BATCH]))
    for m in aqs:
        m.calibrating = False
        m.ready = True
        m.scale.data.fill_(float(m.amax / m.qmax))


def qat_train(model, xt, y, steps, seed=SEED):
    """QAT: train only the activation steps and the head, with the quantizers in the loop.

    The backbone weights stay frozen. Returns the loss of every step.
    """
    torch.manual_seed(seed)
    g = torch.Generator().manual_seed(seed)                # a fixed order of mini-batches
    for p in model.parameters():                            # freeze everything ...
        p.requires_grad = False
    scales = [m.scale for m in model.backbone.modules() if isinstance(m, ActQuantizer)]
    for s in scales:                                        # ... except the 48 steps
        s.requires_grad = True
    for p in model.head.parameters():                       # ... and the head
        p.requires_grad = True
    opt = torch.optim.AdamW([
        {"params": scales, "lr": 5e-4},
        {"params": list(model.head.parameters()), "lr": 1e-3},
    ])
    model.train()
    hist = []
    for step in range(1, steps + 1):
        idx = torch.randint(0, len(y), (BATCH,), generator=g)
        loss = F.cross_entropy(model(to_input(xt[idx.numpy()])), y[idx].to(DEV))
        opt.zero_grad()
        loss.backward()
        opt.step()
        hist.append(loss.item())
        if step % 100 == 0:
            print(f"  QAT step {step:4d}  loss {np.mean(hist[-100:]):.3f}")
    model.eval()
    return hist

In [ ]:
# Bit widths of the weights and of the activations; 8 and 8 give W8A8 (see "If you finish early")
W_BITS, A_BITS = 8, 8

# Two quantized models, each from a fresh copy of the pretrained backbone:
#   PTQ = calibrated only, with the FP32 control's head
#   QAT = calibrated, then the steps and head C trained for QAT_STEPS steps
t0 = time.time()
torch.manual_seed(SEED)
ptq_backbone = quantize_backbone(make_backbone(), W_BITS, A_BITS).to(DEV)
PTQ = Classifier(ptq_backbone, copy.deepcopy(FP_HEAD)).to(DEV)
calibrate(PTQ, TR_X)
qat_backbone = quantize_backbone(make_backbone(), W_BITS, A_BITS).to(DEV)
QAT = Classifier(qat_backbone, copy.deepcopy(HEAD_C)).to(DEV)
calibrate(QAT, TR_X)
LOSS_HIST = qat_train(QAT, TR_X, TR_Y, QAT_STEPS)
print(f"PTQ and QAT ready in {time.time()-t0:.0f}s")

**Evaluation.** Balanced accuracy, the mean of the four per-class recalls, at every blur
level. It counts each class equally, because cars dominate the test crops. The table also gives the
expected calibration error (ECE), the average gap between the model's confidence and its accuracy
(Guo et al., 2017). Lower is better.

In [ ]:
@torch.inference_mode()
def predict(model, xt):
    """Class probabilities (softmax) for uint8 crops, computed in batches."""
    model.eval()
    return torch.cat([torch.softmax(model(to_input(xt[b:b + BATCH])), 1).cpu()
                      for b in range(0, len(xt), BATCH)])


def bal_acc(probs, labels):
    """Balanced accuracy in %: the mean of the four per-class recalls."""
    pred = probs.argmax(1)
    return 100 * float(np.mean([(pred[labels == c] == c).float().mean() for c in range(4)]))


def ece(probs, labels, n_bins=15):
    """Expected calibration error in %, over 15 equal-width confidence bins.

    In each bin, take the gap between the mean confidence and the accuracy; weight it by the
    share of crops in the bin, and add the bins up.
    """
    conf, pred = probs.max(1)
    correct = (pred == labels).float()
    edges = torch.linspace(0, 1, n_bins + 1)
    total = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        in_bin = (conf > lo) & (conf <= hi)
        if in_bin.any():
            total += in_bin.float().mean() * (conf[in_bin].mean() - correct[in_bin].mean()).abs()
    return 100 * float(total)


def evaluate(model):
    """Balanced accuracy and ECE of the model at every blur level, as two {sigma: value} dicts."""
    acc, cal = {}, {}
    for s, xt in TE_CELLS.items():
        probs = predict(model, xt)
        acc[s] = bal_acc(probs, TE_Y)
        cal[s] = ece(probs, TE_Y)
    return acc, cal

In [ ]:
# Balanced accuracy and calibration error of the three models at every blur level
t0 = time.time()
R, E = {}, {}                       # balanced accuracy and ECE, per model and blur level
for name, model in (("FP32", FP32), ("PTQ", PTQ), ("QAT", QAT)):
    R[name], E[name] = evaluate(model)
print(f"evaluated 3 models × {len(SIGMAS)} cells in {time.time()-t0:.0f}s")
print("balanced accuracy and ECE in %, gains in points (pp)\n")

print(f"{'σ':>4} {'FP32':>6} {'PTQ':>6} {'QAT':>6} | {'PTQ−FP32':>9} {'QAT−FP32':>9} | "
      f"{'ECE FP32':>8} {'PTQ':>5} {'QAT':>5}")
for s in SIGMAS:
    gain_ptq = R["PTQ"][s] - R["FP32"][s]
    gain_qat = R["QAT"][s] - R["FP32"][s]
    print(f"{s:>4} {R['FP32'][s]:6.1f} {R['PTQ'][s]:6.1f} {R['QAT'][s]:6.1f} | "
          f"{gain_ptq:+9.1f} {gain_qat:+9.1f} | "
          f"{E['FP32'][s]:8.1f} {E['PTQ'][s]:5.1f} {E['QAT'][s]:5.1f}")

In [ ]:
# Top: accuracy against blur. Middle: the gains of PTQ and QAT over FP32. Bottom: the example.
xs = np.array(SIGMAS)
fp, pt, qa = (np.array([R[k][s] for s in SIGMAS]) for k in ("FP32", "PTQ", "QAT"))
wa = f"W{W_BITS}A{A_BITS}"                      # for example W8A8

fig = plt.figure(figsize=(11, 7))
gs = gridspec.GridSpec(3, 1, height_ratios=[3, 1.6, 1.1], hspace=0.08)
ax0 = fig.add_subplot(gs[0])
ax1 = fig.add_subplot(gs[1], sharex=ax0)
ax2 = fig.add_subplot(gs[2], sharex=ax0)

ax0.plot(xs, fp, "-o", color=C_FP32, lw=2, label="FP32")
ax0.plot(xs, pt, "-o", color=C_PTQ, lw=1.5, label=f"{wa} PTQ (calibrated only)")
ax0.plot(xs, qa, "-o", color=C_QAT, lw=2, label=f"{wa} QAT (trained on clean crops)")
ax0.fill_between(xs, fp, qa, where=qa >= fp, color=C_QAT, alpha=0.25, interpolate=True)     # QAT ahead
ax0.fill_between(xs, fp, qa, where=qa < fp, color="#c0392b", alpha=0.2, interpolate=True)   # QAT behind
ax0.set_ylabel("balanced accuracy (%)")
ax0.legend(loc="lower left", fontsize=9)
ax0.tick_params(labelbottom=False)
ax0.set_title("The real network under blur (one seed, one run)")

ax1.bar(xs - 0.17, pt - fp, width=0.32, color=C_PTQ, label="PTQ − FP32")
ax1.bar(xs + 0.17, qa - fp, width=0.32, color=C_QAT, label="QAT − FP32")
ax1.axhline(0, color="gray", lw=0.8)
ax1.set_ylabel("gain over FP32 (pp)")
ax1.legend(loc="upper left", fontsize=8, ncol=2)
ax1.tick_params(labelbottom=False)

ax2.set_ylim(0, 1)
ax2.grid(False)
ax2.set_yticks([])
ax2.set_ylabel("same object", fontsize=8.5)
ax2.set_xlabel("Gaussian blur σ (pixels on a 224×224 crop)")
for s in SIGMAS:
    ax2.imshow(TE_CELLS[s][EXAMPLE], extent=(s - 0.47, s + 0.47, 0.02, 0.98), aspect="auto", zorder=2)
ax2.set_xlim(-0.8, 14.8)
ax2.set_xticks(xs)
plt.show()

**Read the result.** Compare each quantized model with the FP32 control, one blur level at a time.

* **QAT** costs little on clean crops, gains at moderate blur, and falls behind at the heaviest blur.
  In five runs of this notebook on one Mac GPU, QAT minus FP32 was positive at σ = 4 to 8 in every
  run (+0.6 to +3.9 points), and negative at σ = 10 and 14 in every run. The ICVES paper reports the same shape,
  with its significant gain at σ = 4 to 6 and a reversal at σ = 10 and 14 (Bouguerra et al., 2026b).
* **PTQ** is not a weaker copy of QAT. It loses a few points on clean crops (2.6 in the Mac runs), yet
  it can beat FP32 under blur, by 3.4 points at σ = 4 in the Mac runs. The ICVES paper finds no
  sustained PTQ gain under Gaussian blur, so here runs and pipelines disagree, and the cause is open.
* **The size of each gain varies.** A single run moves by 1 to 3 points with the seed. It also moves
  from run to run, because GPU training is not bit-exact. QAT moved by up to about 3 points between
  runs on the same Mac. And it moves with the hardware, and the vendor's software,
  that run the quantized model. When the presenter's re-run of this recipe (three seeds, not the
  papers' models) was deployed on the neural accelerator of an iPhone, the QAT gain at σ = 6 held
  (+5.6 points, against +5.8 in simulation) while the PTQ gain shrank (+2.1, against +4.1).
* **Calibration.** ECE grows with blur for all three models. In our Mac runs, PTQ's ECE is higher than
  FP32's at most blur levels and QAT's is at or below FP32's at every level. The presenter's re-run
  (three seeds, on A100 GPUs) found QAT about equal to FP32 and PTQ higher.

So read your numbers as one draw. A gain smaller than about 2 points is within the spread, and a
gain measured on one GPU should be measured again on the hardware you deploy on.

*Evidence.* The QAT gain under moderate blur and its reversal under heavy blur are established (the
ICVES paper, and every run here). The PTQ gain, the calibration pattern and the dependence on hardware
are suggestive (a few runs and one re-run).

### 5.2 · General vs specialized features in the real network
Part 4's comparison, done on the 384-dimensional features. Directions come from the
SVD of the clean FP32 features (rank 1 = highest variance). For each rank we measure
the **clean energy**, the **quantization residual** (QAT − FP32 features) and the
**blur residual** (blurred − clean FP32 features). Then the question of Part 4:
how much of each rank's clean information survives quantization, and how much survives blur?

In [ ]:
# 1. Backbone features of the clean and blurred test crops (FP32) and of the clean crops (QAT)
@torch.inference_mode()
def feats(model, xt):
    """Backbone features (without the head) of uint8 crops, as a numpy array."""
    model.eval()
    return torch.cat([model.backbone(to_input(xt[b:b + BATCH])).cpu()
                      for b in range(0, len(xt), BATCH)]).numpy()


F_fp = {s: feats(FP32, TE_CELLS[s]) for s in (0, 2, 6, 14)}
F_q0 = feats(QAT, TE_CELLS[0])

# 2. The ranks: principal directions of the clean FP32 features (rank 1 = most variance)
Xc = F_fp[0] - F_fp[0].mean(0)
_, Sv, Vt = np.linalg.svd(Xc, full_matrices=False)
Vt = Vt[:-1]                     # drop the last direction (the final LayerNorm leaves it no variance)
P0 = Xc @ Vt.T                   # the clean features in the rank basis
sig_e = (P0 ** 2).mean(0)        # clean energy per rank

# 3. Energy per rank of the two residuals: quantization (QAT - FP32) and blur (blurred - clean)
quant_e = (((F_q0 - F_fp[0]) @ Vt.T) ** 2).mean(0)
blur_e = {s: (((F_fp[s] - F_fp[0]) @ Vt.T) ** 2).mean(0) for s in (2, 6, 14)}


# 4. Fraction of each rank's clean information that survives (squared correlation per rank)
def kept(Fd):
    Pd = (Fd - F_fp[0].mean(0)) @ Vt.T
    return np.array([np.corrcoef(P0[:, k], Pd[:, k])[0, 1] ** 2 for k in range(P0.shape[1])])


kept_q = kept(F_q0)
kept_b = {s: kept(F_fp[s]) for s in (2, 6, 14)}
rk = np.arange(1, len(Vt) + 1)


def cosine(a, b):
    return float(a @ b / np.linalg.norm(a) / np.linalg.norm(b))


def smooth(v):
    """Moving average over 9 ranks, for the curves of the middle panel."""
    return np.convolve(v, np.ones(9) / 9, mode="same")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4))

# Left: Part 3's chart on real features, bars coloured by their distance to the quantization residual
K = 60
floor = quant_e[:K].mean()
cols = [C_STRONG if v > 4 * floor else (C_MID if v > floor else C_WEAK) for v in sig_e[:K]]
ax[0].bar(rk[:K], sig_e[:K], color=cols, width=0.8, label="clean feature energy")
ax[0].plot(rk[:K], quant_e[:K], "--", color="#e74c3c", lw=1.5, label="quantization residual (noise floor)")
ax[0].plot(rk[:K], blur_e[6][:K], "-", color=C_FP32, lw=1.5, label="blur residual σ = 6")
ax[0].set(yscale="log", xlabel="feature rank", ylabel="energy",
          title=f"Part 3's chart on real features (first {K} of {len(Vt)} ranks)")
ax[0].legend(fontsize=8)

# Middle: Part 4's bar chart, rank by rank
ax[1].plot(rk, smooth(kept_q), color=C_QAT, lw=2, label=f"after W{W_BITS}A{A_BITS} quantization")
for s, col in ((2, "#9ecae1"), (6, C_FP32), (14, "#08306b")):
    ax[1].plot(rk, smooth(kept_b[s]), color=col, lw=2, label=f"after blur σ = {s}")
ax[1].set(xscale="log", xlabel="feature rank (1 = most variance, 'general')",
          ylabel="fraction of clean information kept", ylim=(0, 1.05),
          title="Part 4's bar chart, rank by rank: strong ranks survive, weak ranks die")
ax[1].legend(fontsize=8)

# Right: per rank, the quantization residual against the blur residual
sc = ax[2].scatter(quant_e, blur_e[6], c=rk, cmap="viridis", s=12, norm=matplotlib.colors.LogNorm())
plt.colorbar(sc, ax=ax[2], label="rank")
ax[2].set(xscale="log", yscale="log", xlabel="quantization residual energy (per rank)",
          ylabel="blur residual energy σ=6 (per rank)",
          title=f"They hit the same ranks: cosine {cosine(quant_e, blur_e[6]):.2f}")
plt.tight_layout()
plt.show()

top, tail = slice(0, 32), slice(128, None)
print(f"information kept, top-32 ranks vs ranks 129+:   "
      f"quantization {kept_q[top].mean():.2f} vs {kept_q[tail].mean():.2f}   "
      f"blur σ=6 {kept_b[6][top].mean():.2f} vs {kept_b[6][tail].mean():.2f}")
print("cosine(quantization residual, blur residual) by σ:",
      {s: round(cosine(quant_e, e), 2) for s, e in blur_e.items()})

**Same picture as the toys, up to a point.** The strong ranks (general, high-variance features)
keep most of their information under both quantization and blur, and the tail ranks (specialized,
low-variance ones) lose most of it under both. Rank by rank, the two residuals point the same way at
mild blur (cosine 0.97 to 0.99 at σ = 2 in our runs) and less so at heavy blur (0.5 to 0.75 at σ = 6
and 14). The very first ranks dip too, because blur also shifts the *mean* feature, and that shift
lands on the top components.

**What this shows, and what it does not.**
* *Established.* Quantization and blur damage the same weak directions of the representation. The
  ICVES paper reports this overlap (Bouguerra et al., 2026b), and the figure above reproduces it at
  small scale.
* *Established.* The size of the noise floor is not the whole story. In the ICVES paper, a model
  trained with noise matched to the quantizers' steps, at the same places, loses about one point under
  blur where QAT gains two to four (Bouguerra et al., 2026b, Table VII). The overlap is necessary, not
  sufficient.
* *Open.* What else in the quantizer does the work. Candidates include clipping, the fact that rounding
  is deterministic, and its dependence on the signal. None has been isolated yet.
* *Suggestive.* How large the gain is depends on the run and on the hardware that executes the model
  (see 5.1).

## Take-aways
1. Quantization noise has a fixed size. Features above it survive, features below it drown (Parts 1–2).
2. Real features have an uneven spectrum, so some are always below the floor. Training under the
   noise moves the decision onto the strong ones (Part 3).
3. Blur and pixel noise remove the same weak features. A model that already ignores them loses less,
   and a model that over-trusts them can fall below it when blur and noise combine (Part 4).
4. In a real transformer, quantization and blur residuals hit the same ranks, and QAT gains under
   moderate blur and loses under the heaviest (Part 5). The size of the gain varies with the run and
   the hardware, and noise of the same size does not reproduce it.

**Exercises.** (a) Part 3 with bits 3, 6 and 8. How does the window move? (b) Part 4 with stronger
stripes (`AMP = 0.15`). When does QAT start reading them too? (c) Replace blur by pixel noise in Part 5.

**If you finish early.** Change `W_BITS` or `A_BITS` in the cell that builds PTQ and QAT (5.1), for
example `A_BITS = 6` or `A_BITS = 4`, then run that cell and every cell after it again. Each new run
takes a few minutes on a GPU. Three cautions apply.
* QAT here trains only the activation steps and the head. The weights keep fixed per-channel steps,
  so a low weight bit width cannot recover.
* Neither paper reports this encoder below 8 bits. The ICVES paper predicts a wider window for coarser
  activations and lists the test among its limitations, so you are running an open experiment.
* One run moves by 1 to 3 points, so trust only larger changes.

**References**
* Bengio, Y., Léonard, N., & Courville, A. (2013). *Estimating or propagating gradients through
  stochastic neurons for conditional computation* (arXiv:1308.3432). arXiv.
  https://arxiv.org/abs/1308.3432
* Bouguerra, A., Montoya Vasquez, D. A., Gomez-Villa, A., Mraidha, C., & Arnez, F. (2026a). Less
  precise can be more reliable: A systematic evaluation of quantization's impact on VLMs beyond
  accuracy. In *Proceedings of the 43rd International Conference on Machine Learning* (Vol. 306).
  PMLR. https://arxiv.org/abs/2509.21173
* Bouguerra, A., Radermacher, A., Arnez, F., & Mraidha, C. (2026b). Quantization and corruption
  robustness in deployed road-scene perception. In *Proceedings of the 2026 IEEE International
  Conference on Vehicular Electronics and Safety (ICVES)*. IEEE. To appear.
* Esser, S. K., McKinstry, J. L., Bablani, D., Appuswamy, R., & Modha, D. S. (2020). Learned step size
  quantization. *International Conference on Learning Representations*.
  https://openreview.net/forum?id=rkgO66VKDS
* Guo, C., Pleiss, G., Sun, Y., & Weinberger, K. Q. (2017). On calibration of modern neural networks.
  In *Proceedings of the 34th International Conference on Machine Learning* (Vol. 70, pp. 1321–1330).
  PMLR. https://proceedings.mlr.press/v70/guo17a.html
* Kenk, M. A., & Hassaballah, M. (2020). *DAWN: Vehicle detection in adverse weather nature dataset*
  (arXiv:2008.05402). arXiv. https://arxiv.org/abs/2008.05402
* Oquab, M., Darcet, T., Moutakanni, T., Vo, H. V., Szafraniec, M., Khalidov, V., Fernandez, P., Haziza,
  D., Massa, F., El-Nouby, A., Assran, M., Ballas, N., Galuba, W., Howes, R., Huang, P.-Y., Li, S.-W.,
  Misra, I., Rabbat, M., Sharma, V., … Bojanowski, P. (2024). DINOv2: Learning robust visual features
  without supervision. *Transactions on Machine Learning Research*.
  https://openreview.net/forum?id=a68SUt6zFt

*Data and model.* The DAWN images (Kenk & Hassaballah, 2020) load from the Hugging Face copy
`Maxim37/dawn-dataset`, licensed CC BY-NC 3.0 (non-commercial use only). The DINOv2 weights load from
`timm/vit_small_patch14_dinov2.lvd142m`, licensed Apache 2.0.

*License.* The code of this notebook is released under the MIT License, copyright 2026 Aymen
Bouguerra and Fabio Arnez.